# Multi-task ResNet18 Attribute Training

Notebook train chung shape/color cho head-tune va last-block fine-tune. Bat Internet trong Kaggle de clone repo, va attach thu muc `data/` co `nih_attribute/`, `splits/`, `processed/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git'
BRANCH = 'CV_attribute_ResNet18_NguyenGiaBao'
REPO_DIR = Path('/kaggle/working/Multiple-Pill-Recognition-And-Interaction-Safety')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

print('Repository:', REPO_DIR)
print('Branch:', BRANCH)

In [ ]:
# Dung cac goi NumPy/SciPy/Pillow co san cua image Kaggle.
# Khong force-reinstall cac goi nen: doi NumPy khi SciPy da duoc kernel nap se gay loi ABI o Cell 5.
numeric_check = subprocess.run(
    [sys.executable, '-c', 'import numpy; import scipy; import sklearn'],
    text=True,
    capture_output=True,
)
if numeric_check.returncode != 0:
    # Chi dung de sua session da bi loi boi cac lan cai dat cu; session moi se khong vao nhanh nay.
    print('Detected broken NumPy/SciPy environment. Repairing compatible numeric packages...', flush=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'uninstall', '-y',
        'numpy', 'scipy', 'scikit-learn'
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2'
    ], check=True)
    print('Numeric packages repaired. Restarting the notebook kernel now...', flush=True)
    import os
    os._exit(0)

# Process con co the doc package moi trong khi kernel hien tai van giu NumPy cu trong RAM.
# Kiem tra lai ngay trong kernel de khong day loi ABI xuong Cell import workflow.
try:
    import numpy as np
    import scipy
    import sklearn
    from scipy.sparse import csr_matrix
except Exception as numeric_kernel_error:
    print(
        'Numeric packages on disk are healthy but this kernel is stale: '
        f'{type(numeric_kernel_error).__name__}: {numeric_kernel_error}',
        flush=True,
    )
    print('Restarting the notebook kernel before workflow imports...', flush=True)
    import os
    os._exit(0)

import PIL
from PIL import Image, ImageDraw
print('Pillow:', PIL.__version__)

# Probe trong process rieng de chua import torch vao kernel notebook.
probe_code = (
    "import torch; "
    "cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (-1,-1); "
    "print('TORCH_PROBE|' + torch.__version__ + '|' + str(torch.version.cuda) + '|' + "
    "str(cap[0]) + '.' + str(cap[1]) + '|' + ','.join(torch.cuda.get_arch_list()))"
)
probe = subprocess.run([sys.executable, '-c', probe_code], text=True, capture_output=True)
probe_line = next(
    (line for line in probe.stdout.splitlines() if line.startswith('TORCH_PROBE|')),
    None,
)

needs_compatible_wheel = probe.returncode != 0 or probe_line is None
if probe_line is not None:
    _, old_torch, old_cuda, capability, arch_text = probe_line.split('|', 4)
    required_arch = 'sm_' + capability.replace('.', '')
    needs_compatible_wheel = required_arch not in arch_text.split(',')
    print('Existing torch:', old_torch, '| CUDA:', old_cuda, '| capability:', capability)
    print('Existing CUDA architectures:', arch_text)

if needs_compatible_wheel:
    print('Current PyTorch wheel does not support this GPU. Installing compatible cu124 wheel...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124'
    ], check=True)
    if 'torch' in sys.modules:
        raise RuntimeError(
            'Da cai PyTorch cu124. Hay Restart Session, sau do Run All de nap wheel moi.'
        )

import torch
import torchvision
if not torch.cuda.is_available():
    raise RuntimeError('Hay bat GPU accelerator trong Kaggle Settings truoc khi train.')

required_arch = 'sm_' + ''.join(map(str, torch.cuda.get_device_capability(0)))
if required_arch not in torch.cuda.get_arch_list():
    raise RuntimeError(
        f'PyTorch {torch.__version__} khong co kernel {required_arch}: ' 
        f'{torch.cuda.get_arch_list()}'
    )

# Smoke test convolution de bat loi kernel ngay tai setup, truoc khi train.
test_conv = torch.nn.Conv2d(3, 4, kernel_size=3).cuda()
test_input = torch.randn(2, 3, 32, 32, device='cuda')
with torch.no_grad():
    test_conv(test_input)
torch.cuda.synchronize()
del test_conv, test_input
torch.cuda.empty_cache()

print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA architectures:', torch.cuda.get_arch_list())
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA convolution smoke test: PASSED')

In [ ]:
# Tu dong tim DATA_ROOT theo cay data/image_all, data/splits va data/processed.
# Khong hard-code slug vi Kaggle co the thay doi so cap thu muc khi mount dataset.
def find_data_root(input_root: Path = Path('/kaggle/input')) -> Path:
    candidates = []
    for image_all_dir in input_root.rglob('image_all'):
        candidate = image_all_dir.parent
        has_images = (image_all_dir / 'nih_attribute/shape').is_dir() and (image_all_dir / 'nih_attribute/color').is_dir()
        has_splits = (candidate / 'splits/nih_attribute/shape').is_dir() and (candidate / 'splits/nih_attribute/color').is_dir()
        if has_images and has_splits and (candidate / 'processed').is_dir():
            candidates.append(candidate)

    unique_candidates = sorted(set(candidates))
    if len(unique_candidates) == 1:
        return unique_candidates[0]
    if not unique_candidates:
        raise FileNotFoundError(
            'Khong tim thay DATA_ROOT. Dataset can co data/image_all/nih_attribute, '
            'data/splits/nih_attribute va data/processed trong /kaggle/input.'
        )
    raise RuntimeError(
        'Tim thay nhieu DATA_ROOT, hay chi dinh mot path: ' +
        ', '.join(str(path) for path in unique_candidates)
    )

DATA_ROOT = find_data_root()
OUTPUT_ROOT = Path('/kaggle/working/attribute_runs')

HEAD_RUN_ID = 'attr_head_v1'
LAST_RUN_ID = 'attr_last_blocks_v1'
RUN_HEAD_TRAIN = True
RUN_HEAD_TEST = True
RUN_LAST_BLOCKS_TRAIN = True
RUN_LAST_BLOCKS_TEST = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
# Preflight: kiem tra schema CSV, source group leakage va synthetic chi nam o train.
from pill_safety.cv.attribute.training.data_contract import validate_attribute_data

paths = {
    'shape_image_dir': DATA_ROOT / 'image_all/nih_attribute/shape',
    'color_image_dir': DATA_ROOT / 'image_all/nih_attribute/color',
    'label_mapping': DATA_ROOT / 'processed/nih_attribute/label_mapping.json',
    'shape_train_csv': DATA_ROOT / 'splits/nih_attribute/shape/train_combined_crop.csv',
    'shape_val_csv': DATA_ROOT / 'splits/nih_attribute/shape/val_combined_crop.csv',
    'shape_test_csv': DATA_ROOT / 'splits/nih_attribute/shape/test_combined_crop.csv',
    'color_train_csv': DATA_ROOT / 'splits/nih_attribute/color/train_multilabel.csv',
    'color_val_csv': DATA_ROOT / 'splits/nih_attribute/color/val_multilabel.csv',
    'color_test_csv': DATA_ROOT / 'splits/nih_attribute/color/test_multilabel.csv',
}
manifest = validate_attribute_data(paths, verify_images=True)
manifest

In [ ]:
from pill_safety.cv.attribute.training.workflow import (
    calibrate_color_thresholds,
    compare_validation_runs,
    evaluate_test,
    load_config,
    train,
)

HEAD_CONFIG = load_config(REPO_DIR / 'configs/training/attribute_resnet18_head_tune/config.yaml')
LAST_CONFIG = load_config(REPO_DIR / 'configs/training/attribute_resnet18_last_blocks_finetune/config.yaml')

In [ ]:
# Phase 1: ImageNet ResNet18, freeze backbone va train shape_head/color_head.
head_checkpoint = OUTPUT_ROOT / 'attribute_resnet18_head_tune/checkpoints' / f'{HEAD_RUN_ID}_best.pt'
if RUN_HEAD_TRAIN:
    head_result = train(HEAD_CONFIG, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    head_checkpoint = Path(head_result['checkpoint'])
if not head_checkpoint.is_file():
    raise FileNotFoundError(f'Head checkpoint not found: {head_checkpoint}')
head_result = {'checkpoint': str(head_checkpoint)}
head_result

In [ ]:
# Calibration cua head: quet threshold TREN validation, luu rieng theo checkpoint.
head_threshold_result = calibrate_color_thresholds(HEAD_CONFIG, head_checkpoint, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
head_thresholds = Path(head_threshold_result['path'])
head_threshold_result

In [ ]:
# Test chi la reporting sau khi head checkpoint va threshold da duoc chon bang validation.
if RUN_HEAD_TEST:
    head_test = evaluate_test(HEAD_CONFIG, head_checkpoint, head_thresholds, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    print(head_test['metrics'])

In [ ]:
# Phase 2: bat buoc nap head best checkpoint, unfreeze layer3/layer4 va hai heads.
last_checkpoint = OUTPUT_ROOT / 'attribute_resnet18_last_blocks_finetune/checkpoints' / f'{LAST_RUN_ID}_best.pt'
if RUN_LAST_BLOCKS_TRAIN:
    last_result = train(LAST_CONFIG, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID, pretrained_override=str(head_checkpoint))
    last_checkpoint = Path(last_result['checkpoint'])
if not last_checkpoint.is_file():
    raise FileNotFoundError(f'Last-block checkpoint not found: {last_checkpoint}')
last_result = {'checkpoint': str(last_checkpoint)}
last_result

In [ ]:
# Calibration va test reporting cua last-block checkpoint.
last_threshold_result = calibrate_color_thresholds(LAST_CONFIG, last_checkpoint, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID)
last_thresholds = Path(last_threshold_result['path'])
if RUN_LAST_BLOCKS_TEST:
    last_test = evaluate_test(LAST_CONFIG, last_checkpoint, last_thresholds, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID)
    print(last_test['metrics'])

In [ ]:
# Chon model theo validation; test khong tham gia quyet dinh.
comparison = compare_validation_runs(
    OUTPUT_ROOT / 'attribute_resnet18_head_tune/metrics' / f'{HEAD_RUN_ID}_val_metrics.json',
    OUTPUT_ROOT / 'attribute_resnet18_last_blocks_finetune/metrics' / f'{LAST_RUN_ID}_val_metrics.json',
    OUTPUT_ROOT / 'attribute_model_selection.json',
)
comparison